In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
1,model_16_2_0,0.150377,0.033872,-0.820822,0.311527,0.134844,1.391265,1.582043,0.552054,1.118602,0.835328,1.406602,1.179519,1.416142,1.229734,145.339573,234.317508,"Hidden Size=[18], regularizer=0.05, learning_r..."
2,model_4_3_0,0.139176,0.105522,-2.917857,-0.018868,0.056203,1.409607,1.464717,0.286078,7.097564,3.691821,1.000937,1.187269,1.558372,1.237813,121.313378,195.664803,"Hidden Size=[15], regularizer=0.05, learning_r..."
3,model_24_5_0,0.136159,-0.026692,-1.168702,0.609099,0.452014,1.414547,1.681217,0.378661,0.634388,0.506525,1.923495,1.189347,1.363722,1.239980,161.306381,260.035323,"Hidden Size=[20], regularizer=0.05, learning_r..."
5,model_10_0_0,0.120333,0.014995,-8.565592,0.020720,0.068949,1.440463,1.612954,0.386341,1.247922,0.817131,0.994807,1.200193,1.469156,1.251287,137.270071,221.372503,"Hidden Size=[17], regularizer=0.2, learning_ra..."
6,model_10_6_0,0.110279,0.062096,-10.487412,0.063837,-0.014581,1.456926,1.535826,0.250439,1.316375,0.783407,1.083513,1.207032,1.474518,1.258418,137.247343,221.349775,"Hidden Size=[17], regularizer=0.2, learning_ra..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,model_10_4_0,-0.130377,-0.077667,-0.441573,-0.121818,-0.052876,1.851004,1.764690,0.431146,7.495088,3.963117,1.349849,1.360516,1.602868,1.418436,136.768544,220.870976,"Hidden Size=[17], regularizer=0.2, learning_ra..."
287,model_8_4_0,-0.134876,-0.076542,-1.096203,-0.155243,-0.053457,1.858370,1.762847,0.413835,7.526413,3.970124,1.425066,1.363220,1.664318,1.421256,128.760600,207.987529,"Hidden Size=[16], regularizer=0.05, learning_r..."
288,model_18_0_0,-0.135794,-0.079756,-1.080493,-0.121902,-0.146895,1.859874,1.768110,0.417585,1.534293,0.975939,1.665370,1.363772,1.514322,1.421830,152.758983,246.612422,"Hidden Size=[19], regularizer=0.2, learning_ra..."
289,model_43_1_0,-0.136131,-0.303983,-0.030794,-1.550843,-0.206516,1.860424,2.135285,1.715622,0.585908,1.150765,1.794352,1.363974,1.354119,1.422041,200.758391,323.864849,"Hidden Size=[25], regularizer=0.05, learning_r..."
